In [ ]:
from chggen.common.sample_utils import CSP_Generator
from chggen.common.data_utils import mkdir

from types import SimpleNamespace
import numpy as np

from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.core import Structure, Composition, Element, Lattice
from pymatgen.io.cif import CifWriter

import pandas as pd
import os
import time
from datetime import datetime

In [ ]:
### Define the kwargs for the generation ###
ld_kwargs = SimpleNamespace(
        n_step_each = 5,            # Corrector
        min_sigma = 0.01,
        num_noise_level = 200,
        signal_to_noise_ratio = 0.4,
        save_traj = False,
        disable_bar = False,
    )
gen_kwargs = SimpleNamespace(
        # num_gen = 3, # number of structures generated from the cubic lattice (not used)
        # num_mutation = 2, # number of mutations during the relax-generation iteration (not used)
        num_cell = 1, # number of times to the formula
        # ehull_cutoff = 0.06, # e_hull cutoff (not used)
        )

device = 'cuda:0'

### Define the CSP file and patched_phase diagram ###
csp = CSP_Generator(chggen_path = "../models/cut_7_conv_3_epoch=27-val_loss=0.87.ckpt",
                    device= device)

In [ ]:
s0 = Structure.from_file("./MnP2O7.cif")

In [ ]:
NUM_GEN = 10 # repeat the number of structures to be generated
host_structure_list = [s0] * NUM_GEN  
num_intercalat_list = [1] * NUM_GEN  # define one Li to be inpainted

In [ ]:
s_list_inpaint = csp.generate_from_host_structure(host_structure_list= host_structure_list,
                                 num_intercalant_list= num_intercalat_list,
                                 ld_kwargs=ld_kwargs, 
                                 species= "Li")

In [ ]:
mkdir("./s_inpaint_structures")
for ii, s_inpaint in enumerate(s_list_inpaint):
    s_inpaint.to(filename = "s_inpaint_structures/LiMnP2O7_"+str(ii)+".cif")